# U-only Score Test

Dedicated standalone test of U-only scoring (rank by `Û`) across all Phase 2 models and k values.
Compared against the Phase 1 P-only baseline.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'src'))
from krauss.backtest.ranking import rank_and_select
from krauss.backtest.portfolio import build_daily_portfolios, aggregate_portfolio_returns
from krauss.backtest.costs import compute_turnover, apply_transaction_costs

ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'

pred2 = pd.read_parquet(PROCESSED / 'predictions_phase2.parquet')
pred1 = pd.read_parquet(PROCESSED / 'predictions_phase1.parquet')
returns = pd.read_parquet(PROCESSED / 'daily_returns.parquet')

pred2['date'] = pd.to_datetime(pred2['date'])
pred1['date'] = pd.to_datetime(pred1['date'])
returns['date'] = pd.to_datetime(returns['date'])

print(f'P2 rows: {len(pred2):,}  dates: {pred2["date"].nunique():,}')
print('U columns:', [c for c in pred2.columns if c.startswith('u_')])

P2 rows: 2,852,210  dates: 5,750
U columns: ['u_rf', 'u_xgb', 'u_dnn', 'u_ens1']


In [2]:
# Sanity: how do the Û predictions look?
u_summary = pred2[['u_dnn', 'u_xgb', 'u_rf', 'u_ens1']].describe().T[['mean', 'std', 'min', 'max']]
display(u_summary.style.format('{:.4f}').set_caption('Û prediction distribution'))

# Correlation of Û with realized U
# realized U = next-day return - cross-sectional median next-day return
realized = pred2.merge(returns, on=['date', 'permno'], how='left')
realized['next_ret'] = realized.groupby('permno')['ret'].shift(-1)
realized['median_next'] = realized.groupby('date')['next_ret'].transform('median')
realized['u_real'] = realized['next_ret'] - realized['median_next']
realized_clean = realized.dropna(subset=['u_real'])

corr_rows = []
for f in ['dnn', 'xgb', 'rf', 'ens1']:
    corr_rows.append({'Model': f.upper(),
                      'corr(Û, U)': realized_clean[f'u_{f}'].corr(realized_clean['u_real'])})
display(pd.DataFrame(corr_rows).set_index('Model').style.format('{:.4f}')
        .set_caption('Correlation of Û predictions with realized U'))

,mean,std,min,max
u_dnn,0.0005,0.0005,-0.0070,0.0038
u_xgb,0.0004,0.0015,-0.0628,0.0538
u_rf,0.0003,0.0044,-0.3060,0.3076
u_ens1,0.0004,0.0017,-0.1029,0.1057


,"corr(Û, U)"
Model,
DNN,-0.0008
XGB,0.0275
RF,0.0071
ENS1,0.0142


In [3]:
def run_backtest(predictions, score_col, k, returns_df, cost_bps=5):
    sel = rank_and_select(predictions, k=k, score_col=score_col)
    hold = build_daily_portfolios(sel, returns_df, k=k)
    daily = aggregate_portfolio_returns(hold)
    turn = compute_turnover(hold, k=k)
    daily = apply_transaction_costs(daily, turn, cost_bps)
    daily['date'] = pd.to_datetime(daily['date'])
    return daily

def sharpe(daily, col):
    r = daily[col]
    std = r.std()
    return (r.mean() / std) * np.sqrt(252) if pd.notna(std) and std > 0 else np.nan

K_VALUES = [10, 50, 100, 150, 200]
rows = []

for family, p1_col in [('dnn', 'p_dnn'), ('xgb', 'p_xgb'), ('rf', 'p_rf'), ('ens1', 'p_ens1')]:
    m = family.upper()
    for k in K_VALUES:
        # P1 baseline (rank by P̂)
        d = run_backtest(pred1, p1_col, k, returns)
        rows.append({'Model': m, 'Score': 'P1 Base', 'k': k,
                     'ret/day': d['port_ret'].mean(),
                     'ret_net/day': d['port_ret_net'].mean(),
                     'Sharpe_net': sharpe(d, 'port_ret_net')})
        # P2 U-only (rank by Û)
        d = run_backtest(pred2, f'score_u_{family}', k, returns)
        rows.append({'Model': m, 'Score': 'P2 U', 'k': k,
                     'ret/day': d['port_ret'].mean(),
                     'ret_net/day': d['port_ret_net'].mean(),
                     'Sharpe_net': sharpe(d, 'port_ret_net')})

res = pd.DataFrame(rows)
print('Backtests complete.')

Backtests complete.


In [4]:
# Headline k=10 comparison
k10 = res[res['k'] == 10].pivot(index='Model', columns='Score', values='ret_net/day')
k10 = k10[['P1 Base', 'P2 U']]
k10 = (k10 * 100).round(4)
display(k10.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=1)
        .set_caption('k=10 post-cost daily return (%)'))

k10s = res[res['k'] == 10].pivot(index='Model', columns='Score', values='Sharpe_net')
k10s = k10s[['P1 Base', 'P2 U']].round(3)
display(k10s.style.format('{:.3f}').background_gradient(cmap='RdYlGn', axis=1)
        .set_caption('k=10 post-cost annualized Sharpe'))

Score,P1 Base,P2 U
Model,,
DNN,0.1370,0.0822
ENS1,0.2788,0.2567
RF,0.2709,0.2200
XGB,0.2149,0.1835


Score,P1 Base,P2 U
Model,,
DNN,0.836,0.517
ENS1,2.177,1.720
RF,2.207,1.509
XGB,1.915,1.478


In [5]:
# Full k grid: post-cost daily return (%) by model and score
for m in ['DNN', 'XGB', 'RF', 'ENS1']:
    sub = res[res['Model'] == m].pivot(index='Score', columns='k', values='ret_net/day')
    sub = sub.reindex(['P1 Base', 'P2 U'])
    sub = (sub * 100).round(4)
    display(sub.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=None)
            .set_caption(f'{m}: post-cost daily return (%)'))

k,10,50,100,150,200
Score,,,,,
P1 Base,0.1370,0.0693,0.0357,0.0230,0.0137
P2 U,0.0822,0.0297,0.0110,0.0036,-0.0002


k,10,50,100,150,200
Score,,,,,
P1 Base,0.2149,0.0939,0.0431,0.0228,0.0139
P2 U,0.1835,0.0832,0.0359,0.0144,0.0064


k,10,50,100,150,200
Score,,,,,
P1 Base,0.2709,0.1205,0.0642,0.0411,0.0264
P2 U,0.2200,0.0839,0.0284,0.0104,0.0041


k,10,50,100,150,200
Score,,,,,
P1 Base,0.2788,0.1208,0.0647,0.0422,0.0306
P2 U,0.2567,0.1012,0.0483,0.0231,0.0130


In [6]:
# Same, but Sharpe
for m in ['DNN', 'XGB', 'RF', 'ENS1']:
    sub = res[res['Model'] == m].pivot(index='Score', columns='k', values='Sharpe_net')
    sub = sub.reindex(['P1 Base', 'P2 U']).round(3)
    display(sub.style.format('{:.3f}').background_gradient(cmap='RdYlGn', axis=None)
            .set_caption(f'{m}: post-cost annualized Sharpe'))

k,10,50,100,150,200
Score,,,,,
P1 Base,0.836,0.742,0.507,0.403,0.291
P2 U,0.517,0.312,0.156,0.064,-0.005


k,10,50,100,150,200
Score,,,,,
P1 Base,1.915,1.417,0.834,0.536,0.391
P2 U,1.478,1.263,0.716,0.347,0.184


k,10,50,100,150,200
Score,,,,,
P1 Base,2.207,1.898,1.357,1.066,0.813
P2 U,1.509,1.148,0.508,0.224,0.104


k,10,50,100,150,200
Score,,,,,
P1 Base,2.177,1.623,1.148,0.915,0.795
P2 U,1.720,1.357,0.868,0.510,0.343


In [7]:
# Pre- vs post-cost gap (turnover sanity check)
gap = res.copy()
gap['cost_drag_bps'] = (gap['ret/day'] - gap['ret_net/day']) * 1e4
gap_tbl = gap[gap['k'] == 10].pivot(index='Model', columns='Score', values='cost_drag_bps')
gap_tbl = gap_tbl[['P1 Base', 'P2 U']].round(2)
display(gap_tbl.style.format('{:.2f}').set_caption('k=10 daily cost drag (bps)'))

Score,P1 Base,P2 U
Model,,
DNN,7.13,6.02
ENS1,12.84,11.44
RF,13.62,10.18
XGB,13.84,13.48
